# DRISHTI — full pipeline on a Colab / Kaggle GPU (run EVERYTHING here)

You have the light spine locally but **not** the heavy dependencies (COLMAP, Open3D, Torch, …), so a
local run fails loudly at `s2_poses`. This notebook installs the full stack on a free **T4** and runs
the **entire 11-stage pipeline end-to-end on the GPU** — `s0_ingest → … → report` — then hands you the
whole bundle (model + deliverables + accuracy report + **logs**) to download.

> Sibling notebook `drishti_cloud_t4.ipynb` does the *tier split* (heavy stages on the T4, then
> `resume` the light stages on your machine). **This** notebook does the opposite: **all** stages here,
> nothing left for the ground. Both write the same content-hashed bundle.

**Honest by construction:** it installs the real package, runs the real `drishti` CLI, and prints the
real `doctor` / `inspect` / `verify` output. No stubs, no fabricated numbers — a missing dependency,
GPU, or input **fails loudly** and says exactly what is missing.

### Run order
1. **Runtime → Change runtime type → T4 GPU** (Colab) / enable the GPU accelerator (Kaggle).
2. Set `REPO_URL` in §1.
3. Run §0–§4 (GPU → code → install → doctor).
4. In §5 pick **one** import option (generate here / your own video / resume a partial bundle).
5. Run §6 (full pipeline), then §7–§9 (inspect → logs → download).

## 0 · Confirm the GPU

If no Tesla GPU is listed, stop and switch the runtime to GPU — the neural (S3/S4) and dense (S7) stages
need CUDA.

In [ ]:
!nvidia-smi

## 1 · Get the code

Point these at your DRISHTI repository. On Kaggle you can instead attach the repo as a dataset and set
`REPO_DIR` to its path (e.g. `/kaggle/input/drishti`), then skip the clone.

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-org>/drishti.git"   # <-- set me
REPO_REF = "main"                                         # branch, tag, or commit
REPO_DIR = Path("/content/drishti") if Path("/content").exists() else Path("/kaggle/working/drishti")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"{REPO_DIR} already present — skipping clone")

os.chdir(REPO_DIR)

# Defaults so §6 never NameErrors even if you run only one import cell in §5.
MISSION = ""            # a dataset descriptor .yaml  -> fresh full run
RESUME_BUNDLE = None    # a partial bundle directory  -> resume on the GPU
print("cwd:", Path.cwd())

## 2 · Install DRISHTI + the two packaging gaps

Colab/Kaggle already ship a CUDA build of PyTorch — we keep it. Then we install:

1. the heavy capability groups from `pyproject.toml` (`video, geo, recon, poses, spine, telem, depth, server`), and
2. two packages the extras don't list but the full run genuinely needs — installed explicitly and honestly:
   - **`transformers`** — S3 (RT-DETR) and S4 (Depth Anything V2) load their models from HuggingFace
     `transformers`; the `depth` extra ships only the `torch/openvino/onnx` *runtimes*, not `transformers`.
   - **`laspy`** — S10 writes the **LAS** point cloud (a MUST deliverable); it isn't in any extra either.

Without those two, `doctor` (next) would mark `s3_masking`, `s4_depth`, and `s10_export` blocked.

In [ ]:
# Keep the hosted CUDA torch; do NOT let an extra pull a second copy.
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

# 1) DRISHTI + all heavy capability groups.
!pip install -q -e ".[video,geo,recon,poses,spine,telem,depth,server]"

# 2) Packaging gaps (models + LAS writer) — see the note above.
!pip install -q -U "transformers>=4.45" timm safetensors
!pip install -q "laspy[lazrs]>=2.5"

# 3) Pillow: scripts/make_sample_dataset.py reads GPS EXIF on the real-dataset import path.
!pip install -q pillow

!pip check || echo "(pip check warnings are non-fatal; investigate only if a stage errors)"

## 3 · (optional) Blender — so the FBX deliverable is real, not skipped

**FBX** is a required output. S10 produces it out-of-process with **headless Blender ≥ 4.0** (it calls
`bpy.ops.wm.ply_import`, which exists only in Blender 4.x — the distro's `apt` Blender is too old). This
cell fetches a portable Blender 4.2 LTS and puts it on `PATH`.

Skip this cell if you don't need FBX: it is then recorded as *skipped* and `s10_export` marked
**degraded** (not failed) — every other format still exports.

In [ ]:
import os, shutil, subprocess, urllib.request
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path("/kaggle/working")

def _fetch(url, dest):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "drishti"})
        with urllib.request.urlopen(req, timeout=90) as r:
            if getattr(r, "status", 200) >= 400:
                return False
            with open(dest, "wb") as f:
                shutil.copyfileobj(r, f)
        return dest.stat().st_size > 1_000_000
    except Exception as e:
        print("   ", url.rsplit("/", 1)[-1], "->", e)
        return False

if shutil.which("blender"):
    print("blender already on PATH:", shutil.which("blender"))
else:
    tarxz = BASE / "blender.tar.xz"
    got = None
    for v in ["4.2.5", "4.2.4", "4.2.3", "4.2.1", "4.2.0"]:
        print("trying Blender", v)
        if _fetch(f"https://download.blender.org/release/Blender4.2/blender-{v}-linux-x64.tar.xz", tarxz):
            got = v
            break
    if not got:
        print("\n! Blender not fetched — FBX will be skipped (s10_export degraded). This is non-fatal.")
    else:
        dest = BASE / "blender"
        shutil.rmtree(dest, ignore_errors=True)
        dest.mkdir()
        subprocess.run(["tar", "-xf", str(tarxz), "-C", str(dest), "--strip-components=1"], check=True)
        link = Path("/usr/local/bin/blender")
        if link.exists() or link.is_symlink():
            link.unlink()
        link.symlink_to(dest / "blender")
        print(subprocess.run(["blender", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

## 4 · Doctor — the honest gate (spend GPU minutes only if this is green)

Expect **cuda: yes** and every stage **ready** (all extras + the two gap packages installed; the FBX
tool present if you ran §3). Anything still `blocked` here will fail loudly when reached — fix it first.

In [ ]:
!drishti doctor

## 5 · Provide inputs (IMPORT) — run **ONE** of the next three cells

| Option | Use when | What it sets |
|--------|----------|--------------|
| **D · generate here** | first real run / you have no data of your own | `MISSION` (fresh full run) |
| **A/B · your own mission** | you have a video + telemetry + descriptor | `MISSION` (fresh full run) |
| **C · resume a partial bundle** | you already ran S0/S1 locally and want to continue | `RESUME_BUNDLE` |

A DRISHTI input is a `mission.yaml` **descriptor** plus the files it points at (video + GPS track, and
any optional IMU / baro / intrinsics / RTK). See `docs/GUIDE.md` for the descriptor contract.

In [ ]:
# == IMPORT · Option D — GENERATE a realistic dataset on this GPU (no upload) ==================
# Assembles the real MP4+SRT contract from open, GPS-tagged aerial imagery (or a ground-truth
# synthetic city). Only playback timing is synthesized; pixels + GPS are the source's own.
DATASET = "aukerman"   # aukerman(real buildings ~543MB) | brighton_beach(fast ~62MB) | caliterra | lewis | synthetic

if DATASET == "synthetic":
    !python scripts/make_sample_dataset.py --synthetic
    _name = "synthetic_city"
else:
    !python scripts/make_sample_dataset.py --dataset {DATASET}
    _name = DATASET

import os
from pathlib import Path
MISSION = str(REPO_DIR / "configs" / "datasets" / f"{_name}.yaml")
RESUME_BUNDLE = None
os.environ["DRISHTI_MISSION"] = MISSION
assert Path(MISSION).is_file(), MISSION
print("\nMISSION =", MISSION, "\n")
print(Path(MISSION).read_text())

In [ ]:
# == IMPORT · Options A/B — bring YOUR OWN mission (fresh full run) =============================
# Uncomment ONE. Point at the descriptor AND make sure the files it references are present.
from pathlib import Path

# A) Google Drive (best for a large video):
# from google.colab import drive; drive.mount("/content/drive")
# MISSION = "/content/drive/MyDrive/drishti/my_mission.yaml"

# B) Direct upload of small files (descriptor + video + telemetry together):
# from google.colab import files; up = files.upload()
# MISSION = next(k for k in up if k.endswith((".yaml", ".yml")))

RESUME_BUNDLE = None
assert MISSION and Path(MISSION).is_file(), (
    f"MISSION not set/found ({MISSION!r}). Uncomment A or B above, or use Option D.")
print("mission:", MISSION)

In [ ]:
# == IMPORT · Option C — RESUME a partial bundle on the GPU ====================================
# You ran S0/S1 locally and S2 failed for lack of GPU deps. Continue that exact bundle here.
#
# IMPORTANT: the runner re-checks S0's inputs, so the descriptor's video + telemetry must exist at
# their recorded relative paths. Either regenerate them (if they came from make_sample_dataset — same
# paths) or upload/mount your originals BEFORE resuming. If the files match what the local run hashed,
# S0/S1 are skipped (fresh); otherwise they recompute — either way it runs through to completion.
RUN_OPTION_C = False   # <-- set True to use this cell

if RUN_OPTION_C:
    import zipfile
    from pathlib import Path
    # (a) make the ORIGINAL inputs available at their recorded paths, e.g. regenerate a sample set:
    # !python scripts/make_sample_dataset.py --dataset brighton_beach
    # (b) upload the partial bundle you zipped locally (PowerShell:  Compress-Archive runs\<id> <id>.zip)
    RUNS_DIR = REPO_DIR / "runs"
    RUNS_DIR.mkdir(exist_ok=True)
    from google.colab import files
    up = files.upload()
    zname = next(k for k in up if k.endswith(".zip"))
    with zipfile.ZipFile(zname) as z:
        z.extractall(RUNS_DIR)
    cand = sorted(RUNS_DIR.rglob("manifest.json"), key=lambda p: p.stat().st_mtime)
    assert cand, "no manifest.json found after unzip — zip the run_id FOLDER (not the runs/ parent)."
    RESUME_BUNDLE = str(cand[-1].parent)
    MISSION = ""
    print("will RESUME:", RESUME_BUNDLE)
else:
    print("Option C disabled (RUN_OPTION_C = False). Using Option D/A/B's MISSION.")

## 6 · Run the FULL pipeline (all 11 stages, on the GPU)

Uses the **`balanced`** profile, which runs end-to-end here: `s0_ingest → … → report`.

> Why not `--profile max`? `max` selects `dense.method=gaussian` + `mesh.method=2dgs`, which this build
> does not implement — those stages **raise loudly** (honest; no silent fallback). For higher quality on
> the implemented (TSDF + Poisson) path, keep `balanced`, or add
> `--set dense.method=tsdf --set mesh.method=poisson` on top of `--profile max`.

`DRISHTI_LOG_LEVEL=DEBUG` gives verbose logs, and the **entire console is tee'd** into the bundle at
`logs/run_console.log` so it downloads with everything else. The first run also fetches the RT-DETR and
Depth-Anything-V2 (base) checkpoints (both Apache-2.0) into the model cache.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

os.environ["DRISHTI_LOG_LEVEL"] = "DEBUG"                 # verbose, honest logs
os.environ["DRISHTI_RUNS_DIR"] = str(REPO_DIR / "runs")   # where inspect/verify/server look, too
PROFILE = "balanced"

if RESUME_BUNDLE:
    bundle_path = Path(RESUME_BUNDLE)
    cmd = ["drishti", "resume", str(bundle_path)]
else:
    assert MISSION and Path(MISSION).is_file(), "no MISSION — run an import cell in §5 first."
    run_id = time.strftime("colab-full-%Y%m%d-%H%M%S")
    bundle_path = REPO_DIR / "runs" / run_id
    (bundle_path / "logs").mkdir(parents=True, exist_ok=True)   # so we can tee before the run starts
    cmd = ["drishti", "run", "--dataset", MISSION, "--profile", PROFILE, "--run-id", run_id]

logfile = bundle_path / "logs" / "run_console.log"
print(">>", " ".join(cmd))
print("   console tee ->", logfile, "\n")

with open(logfile, "w", encoding="utf-8") as lf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
        lf.write(line)
    proc.wait()

BUNDLE = str(bundle_path)
print(f"\n[exit {proc.returncode}]  bundle: {BUNDLE}")
if proc.returncode != 0:
    print("A stage failed above — that is the honest, fail-loud behaviour. Inspect it in §7, fix the "
          "cause, then set  RESUME_BUNDLE = BUNDLE  and re-run this cell; completed stages are skipped.")

## 7 · Inspect + verify (the ground truth)

`inspect` prints the manifest — per-stage status, where it ran, wall-time, **measured** confidence, and
any degradation. `verify` recomputes every output hash and checks it against the manifest.

In [ ]:
!drishti inspect "$BUNDLE"
!drishti verify "$BUNDLE"

## 8 · Logs & report

Everything the run recorded: the tee'd console log, the per-stage manifest (above), and the accuracy
report the final stage wrote.

In [ ]:
import json
from pathlib import Path
B = Path(BUNDLE)

print("=== logs/ ===")
for p in sorted((B / "logs").glob("*")):
    print(f"  {p.name:24} {p.stat().st_size:>10,} B")

rc = B / "logs" / "run_console.log"
if rc.is_file():
    print("\n=== last 25 lines of run_console.log ===")
    print("".join(rc.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)[-25:]))

print("=== report/ ===")
for p in sorted((B / "report").glob("*")):
    print("  ", p.name)

rj = B / "report" / "report.json"
if rj.is_file():
    r = json.loads(rj.read_text(encoding="utf-8"))
    print("\n=== report.json (summary) ===")
    for k in ("drishti_version", "total_wall_seconds", "n_failed", "n_degraded"):
        if k in r:
            print(f"  {k}: {r[k]}")
    if "accuracy" in r:
        print("  accuracy:", json.dumps(r["accuracy"]))
    print("\nOpen report/report.html locally for the full formatted report.")

## 9 · Export EVERYTHING — download the whole bundle (incl. logs)

Two archives:
- **full** — the complete bundle: `manifest.json`, `inputs/`, `logs/`, `report/`, and every stage dir
  (raw frames, depth, dense cloud, mesh, and all `s10_export` deliverables). Everything, reproducible.
- **slim** — just deliverables + report + logs + manifest (no raw inputs/intermediates), for a quick share.

Big real sets (aukerman/lewis) make the full zip large; if a browser download stalls, copy it to Google
Drive instead (commented below).

In [ ]:
import shutil, zipfile
from pathlib import Path
B = Path(BUNDLE)
rid = B.name

# --- full bundle ---
full_zip = shutil.make_archive(str(B.parent / f"{rid}_full"), "zip", root_dir=B.parent, base_dir=rid)
print("full :", full_zip, f"({Path(full_zip).stat().st_size / 1e6:.1f} MB)")

# --- slim: deliverables + report + logs + manifest ---
slim_zip = str(B.parent / f"{rid}_slim.zip")
with zipfile.ZipFile(slim_zip, "w", zipfile.ZIP_DEFLATED) as z:
    if (B / "manifest.json").is_file():
        z.write(B / "manifest.json", f"{rid}/manifest.json")
    for sub in ["report", "logs", "s10_export"]:
        for p in (B / sub).rglob("*"):
            if p.is_file():
                z.write(p, f"{rid}/{p.relative_to(B)}")
print("slim :", slim_zip, f"({Path(slim_zip).stat().st_size / 1e6:.1f} MB)")

# --- download (Colab) ---
try:
    from google.colab import files
    files.download(slim_zip)      # small first; swap to full_zip for everything
    # files.download(full_zip)
except Exception:
    print("Not on Colab — grab the archives from the file browser (Kaggle: /kaggle/working).")

# --- or copy to Drive for large bundles ---
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy2(full_zip, "/content/drive/MyDrive/")

## 10 · Round-trip back to your machine

You ran the whole pipeline here, so locally you only need to **view / serve** it — no heavy deps required:

```bash
unzip <run_id>_full.zip -d runs/
drishti inspect runs/<run_id>            # the same manifest you saw above
drishti verify  runs/<run_id>            # confirm the download is intact

pip install -e ".[server]"
python -m server.app                     # http://localhost:8000/api/health
cd viewer && npm install && npm run dev   # browse the georeferenced model
```

`drishti resume runs/<run_id>` is a no-op once every stage is `done` (it just re-checks freshness). The
`report/report.html` in the bundle is the shareable accuracy report; `s10_export/` holds the
OBJ · PLY · LAS · GeoTIFF · glTF · FBX deliverables.